# 01. 보정 NPZ 생성 (카메라 위치를 바꿨을 때 1회)

다이어그램 입력 블록 **"보정값 NPZ: R, t"** 를 만드는 노트북이다.
ArUco 마커(ID0, 바닥) 4개 코너로 각 카메라의 외부 파라미터(R, t)를 구해 NPZ로 저장한다.
K, D(렌즈 내부 파라미터)는 config.yaml의 `cameras` 값을 쓴다.

| 항목 | 내용 |
|---|---|
| 입력 | 마커가 보이는 상단·정면 영상의 한 프레임 (`calibration.*`, 비어 있으면 `inputs.*_video`) |
| 출력 | `inputs.calib_top_npz`, `inputs.calib_front_npz` 경로의 NPZ (02·03이 같은 경로를 읽음) |
| GPU | 필요 없음 |
| 바꿀 것 | 1번 셀의 `CONFIG_PATH` 하나. 나머지 값은 config.yaml에서 관리 |

**언제 다시 실행하나:** 카메라를 옮겼거나 수조·마커 위치가 바뀌었을 때. 같은 셋업이면 이전 NPZ를 그대로 쓴다.

**순서:** 자동인식 → 좌표축 확인 → (틀리면) 클릭 또는 코너 순서 회전 → 저장. 상단과 정면을 각각 한다.

In [ ]:
#@title 0. 환경 준비 (저장소 받기 · 라이브러리 설치 · Drive 연결)
REPO_URL = "https://github.com/YOUR_ID/underwater-gripper-tracking.git"  #@param {type:"string"}

import os, sys, subprocess, warnings
REPO_DIR = '/content/' + REPO_URL.rstrip('/').split('/')[-1].removesuffix('.git')
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=False)
%pip install -q -r {REPO_DIR}/requirements.txt

from google.colab import drive
drive.mount('/content/drive')

sys.path.insert(0, REPO_DIR)
import pandas as pd
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
print('준비 완료:', REPO_DIR)

In [ ]:
#@title 1. 설정 파일 지정 · 점검
CONFIG_PATH = "/content/drive/MyDrive/8월연구/config.yaml"  #@param {type:"string"}

import shutil
from gtrack import settings

if not os.path.exists(CONFIG_PATH):
    os.makedirs(os.path.dirname(CONFIG_PATH), exist_ok=True)
    shutil.copy(os.path.join(REPO_DIR, 'config.yaml'), CONFIG_PATH)
    raise RuntimeError(
        f'설정 파일이 없어 템플릿을 복사했습니다: {CONFIG_PATH}\n'
        '왼쪽 파일 패널에서 이 파일을 더블클릭해 [필수] 항목을 수정한 뒤 이 셀을 다시 실행하세요.')

settings.configure(CONFIG_PATH)
settings.print_summary('calibration')
settings.check('calibration')

## 상단 카메라

**2-1.** 프레임을 읽고 ArUco 자동인식을 시도한 뒤 좌표축을 그린다.
확인 기준: **X(빨강) 오른쪽 / Y(초록) 수조 안쪽 / Z(파랑) 위**.

In [ ]:
#@title 2-1. 상단: 프레임 읽기 · 자동인식 · 좌표축 확인
from gtrack.calibration.core import MarkerCalibrator, CORNER_LABELS
from gtrack.calibration.colab_click import click_points

top = MarkerCalibrator('top')
top.load_frame()
if settings.CALIB_AUTO_DETECT:
    top.auto_detect()
if top.points:
    top.preview()

**2-2. (자동인식이 실패했거나 코너가 어긋났을 때만)** `MANUAL_CLICK`을 체크하고 실행해 직접 클릭한다.
1) 큰 화면에서 코너 근처를 클릭 → 2) 오른쪽 확대경 안을 클릭해 정확한 위치로 옮김(방향키로 0.25 px 미세 이동) → 3) [점 확정].
순서: **1=좌상단(TL) → 2=우상단(TR) → 3=우하단(BR) → 4=좌하단(BL)**. 4점을 확정하면 [완료].

클릭 화면이 뜨지 않으면 아래처럼 픽셀 좌표를 직접 넣어도 된다: `top.set_points([[u1,v1],[u2,v2],[u3,v3],[u4,v4]])`

In [ ]:
#@title 2-2. 상단: 직접 클릭 (필요할 때만 체크)
MANUAL_CLICK = False  #@param {type:"boolean"}
if MANUAL_CLICK:
    pts = click_points(top.frame, n_points=4, labels=CORNER_LABELS)
    top.set_points(pts)
    top.preview()
elif not top.points:
    print('⚠️ 자동인식 결과가 없습니다. MANUAL_CLICK을 체크하고 이 셀을 다시 실행하세요.')
else:
    print('자동인식 결과를 사용합니다.')

In [ ]:
#@title 2-3. 상단: 축 방향이 틀릴 때 코너 순서 회전 (맞으면 0 유지)
ROTATE_STEPS = 0  #@param {type:"integer"}
if ROTATE_STEPS:
    top.rotate_order(ROTATE_STEPS)
    top.preview()

In [ ]:
#@title 2-4. 상단: NPZ 저장
OVERWRITE = False  #@param {type:"boolean"}
top.save(overwrite=OVERWRITE)

**2-5. (선택) 측정점 확인.** `MEASURE`를 체크하고 실행한 뒤 한 점을 클릭하면 마커 중심 기준 X, Y, Z와 거리를 출력하고 NPZ에 추가 저장한다.
그 점이 마커 평면이 아닌 높이에 있으면 `TARGET_Z_CM`을 그 높이로 바꾼다. 굴절은 고려하지 않으므로 마른 상태 영상에서만 의미가 있다.

In [ ]:
#@title 2-5. 상단: 측정점 (선택)
MEASURE = False  #@param {type:"boolean"}
TARGET_Z_CM = 0.0  #@param {type:"number"}
if MEASURE:
    p = click_points(top.preview_image()[0], n_points=1, labels=['측정점'])[0]
    top.measure_point(float(p[0]), float(p[1]), target_z=TARGET_Z_CM)
    top.save_measured_point()

## 정면 카메라

정면은 **어떤 마커를 찍느냐**에 따라 처리가 다르다 (config.yaml `calibration.front_use_wall_transform`).
- `false` : 수조 내부 바닥 ID0 마커를 정면에서 찍음 → 찍은 마커가 곧 world
- `true`  : 수조 밖 벽 마커를 찍음 → `setup.front_wall_marker_R/t`로 world 변환

같은 ID 마커가 두 개 보이면 `front_pick_rightmost: true`일 때 오른쪽 마커를 고른다.

In [ ]:
#@title 3-1. 정면: 프레임 읽기 · 자동인식 · 좌표축 확인
from gtrack.calibration.core import MarkerCalibrator, CORNER_LABELS
from gtrack.calibration.colab_click import click_points

front = MarkerCalibrator('front')
front.load_frame()
if settings.CALIB_AUTO_DETECT:
    front.auto_detect()
if front.points:
    front.preview()

In [ ]:
#@title 3-2. 정면: 직접 클릭 (필요할 때만 체크)
MANUAL_CLICK = False  #@param {type:"boolean"}
if MANUAL_CLICK:
    pts = click_points(front.frame, n_points=4, labels=CORNER_LABELS)
    front.set_points(pts)
    front.preview()
elif not front.points:
    print('⚠️ 자동인식 결과가 없습니다. MANUAL_CLICK을 체크하고 이 셀을 다시 실행하세요.')
else:
    print('자동인식 결과를 사용합니다.')

In [ ]:
#@title 3-3. 정면: 축 방향이 틀릴 때 코너 순서 회전 (맞으면 0 유지)
ROTATE_STEPS = 0  #@param {type:"integer"}
if ROTATE_STEPS:
    front.rotate_order(ROTATE_STEPS)
    front.preview()

In [ ]:
#@title 3-4. 정면: NPZ 저장
OVERWRITE = False  #@param {type:"boolean"}
front.save(overwrite=OVERWRITE)

## 4. 최종 확인

저장한 두 NPZ를 전처리(02)와 같은 방식으로 읽어서, 두 카메라에 **모터 중심(출력 원점)** 과 **world 축**을 투영한다.
두 화면에서 흰 점이 실제 그리퍼 중심에 오고, 축 방향이 같으면 두 카메라의 좌표계가 맞게 통합된 것이다.

In [ ]:
#@title 4. 두 카메라 좌표계 통합 확인
import cv2, matplotlib.pyplot as plt
from gtrack.online import calibration_io
from gtrack.online.visualize import draw_overlay

calib_top = calibration_io._load_top_calib(settings.MANUAL_CALIB_TOP_PATH)
calib_front, use_wall = calibration_io._load_front_calib(settings.MANUAL_CALIB_FRONT_PATH)
calibration_io._calib_summary('top', calib_top)
calibration_io._calib_summary('front', calib_front)

fig, axes = plt.subplots(1, 2, figsize=(18, 5.5))
for ax, cal, calib, is_front in ((axes[0], top, calib_top, False), (axes[1], front, calib_front, True)):
    img = draw_overlay(cal.frame.copy(), calib, is_front=is_front)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.set_title(cal.role); ax.axis('off')
plt.tight_layout(); plt.show()
print('흰 점 = 모터 중심, 자홍 사각형 = 작업영역(그리퍼 주변 50 cm)')